# 03 · Líneas base y candidatos

El registro en MLflow y la inferencia de TimesFM se habilitan en la primera celda. El diagnóstico temporal y la comparación usan el conjunto de desarrollo.

In [ ]:
REGISTRAR_CANDIDATOS = False
RECALCULAR_TIMESFM = False
REGISTRAR_TIMESFM = False

In [33]:
import json
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

directorio = Path.cwd().resolve()
raiz = next(
    (
        ruta for ruta in [directorio, *directorio.parents]
        if (ruta / "src" / "pipeline.py").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio pset3.")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42
HORIZONTE = 5
CUANTIL_TARGET = 0.80
inicio_prueba = pd.Timestamp("2023-07-11")
capital_inicial = 10_000.0

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")
cliente = MlflowClient()
NOMBRE_EXPERIMENTO = "Sistema de alerta temprana de volatilidad para acciones"
NOMBRE_REGISTRO = "alerta-volatilidad-aapl-modelo"
ruta_revision = raiz / "docs" / "revision_runs.json"

def guardar_ids(contenido):
    temporal = ruta_revision.with_suffix(".tmp")
    ruta_revision.parent.mkdir(exist_ok=True)
    temporal.write_text(json.dumps(contenido, indent=2), encoding="utf-8")
    temporal.replace(ruta_revision)

revision = (
    json.loads(ruta_revision.read_text(encoding="utf-8"))
    if ruta_revision.is_file() else {}
)

In [34]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '673213bc5b0fe2ef795cd2492ece28723d4b3d92656aadfb5cadb71ed13dbafb', 'filas_snapshot': 4087}


In [35]:
features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(HORIZONTE, min_periods=HORIZONTE)
    .std(ddof=1)
    .shift(-HORIZONTE)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-HORIZONTE)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

fechas_holdout = datos_modelado.index[datos_modelado.index >= inicio_prueba]

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

# Retorno de t a t+1, expresado como proporción y alineado con t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

assert fin_ventana_target.loc[X_desarrollo.index].max() < fechas_holdout.min()

print("Desarrollo:", X_desarrollo.shape)
print("Fechas reservadas para prueba:", len(fechas_holdout))

Desarrollo: (3212, 18)
Fechas reservadas para prueba: 805


Separación temporal y horizonte de predicción. Se usaron cinco pliegues de validación temporal con una separación de cinco ruedas bursátiles (gap=5) entre el final del entrenamiento y el inicio de la validación de cada pliegue. La etiqueta de una alerta emitida al cierre de la rueda t se calcula con los rendimientos de las ruedas t+1 a t+5. Por ello, se excluyen del entrenamiento las cinco observaciones inmediatamente anteriores al inicio de la validación. Así, la etiqueta de la última observación de entrenamiento se completa antes de la primera rueda de validación. Para el hold-out se aplica el mismo criterio: solo se conservan en entrenamiento las observaciones cuya ventana objetivo termina antes del 11 de julio de 2023.

## Dummy Classifier

In [36]:
#BASELINE DEL DUMMY CLASSIFIER

from time import perf_counter

from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

from src.pipeline import crear_pipeline

resultados_dummy = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    X_train = X_desarrollo.iloc[idx_train]
    X_val = X_desarrollo.iloc[idx_val]

    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    dummy = crear_pipeline(
        DummyClassifier(strategy="most_frequent"),
        columnas_numericas,
    )

    inicio = perf_counter()
    dummy.fit(X_train, y_train)
    segundos_ajuste = perf_counter() - inicio

    proba_train = dummy.predict_proba(X_train)[:, 1]
    proba_val = dummy.predict_proba(X_val)[:, 1]

    resultados_dummy.append({
        "pliegue": pliegue,
        "p80_train": p80,
        "positivos_validacion": int(y_val.sum()),
        "roc_auc_train": roc_auc_score(y_train, proba_train),
        "roc_auc_val": roc_auc_score(y_val, proba_val),
        "segundos_ajuste": segundos_ajuste,
    })

resultados_dummy = pd.DataFrame(resultados_dummy)
display(resultados_dummy.round(4))

,pliegue,p80_train,positivos_validacion,roc_auc_train,roc_auc_val,segundos_ajuste
0,1,1.8833,106,0.5,0.5,0.0915
1,2,1.8899,110,0.5,0.5,0.0156
2,3,1.8949,105,0.5,0.5,0.0212
3,4,1.8921,218,0.5,0.5,0.0152
4,5,2.0642,133,0.5,0.5,0.0172


## Baseline Persistencia

In [37]:
#BASELINE "Random Walk" para volatilidad (persistencia)

resultados_persistencia = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    score_train = X_desarrollo.iloc[idx_train]["VOL_HIST_5"]
    score_val = X_desarrollo.iloc[idx_val]["VOL_HIST_5"]
    alertas_val = score_val.gt(p80)

    resultados_persistencia.append({
        "pliegue": pliegue,
        "p80_train": p80,
        "positivos_validacion": int(y_val.sum()),
        "alertas_validacion": int(alertas_val.sum()),
        "roc_auc_train": roc_auc_score(y_train, score_train),
        "roc_auc_val": roc_auc_score(y_val, score_val),
    })

resultados_persistencia = pd.DataFrame(resultados_persistencia)
display(resultados_persistencia.round(4))

auc_cv = resultados_persistencia["roc_auc_val"]
print(f"ROC-AUC de validación: {auc_cv.mean():.4f} ± {auc_cv.std(ddof=1):.4f}")

,pliegue,p80_train,positivos_validacion,alertas_validacion,roc_auc_train,roc_auc_val
0,1,1.8833,106,109,0.5308,0.5771
1,2,1.8899,110,110,0.5613,0.5978
2,3,1.8949,105,105,0.5702,0.7414
3,4,1.8921,218,216,0.6170,0.7113
4,5,2.0642,133,133,0.6707,0.6989


ROC-AUC de validación: 0.6653 ± 0.0731


# Baseline de Negocio: Buy and Hold

In [38]:
capital_inicial = 10_000.0

# RET_LAG_0 está en porcentaje. Alinear el próximo retorno con la fecha t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

resultados_buy_hold = []

for pliegue, (_, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    fechas_val = X_desarrollo.index[idx_val]
    retornos = retorno_siguiente.loc[fechas_val]

    assert retornos.notna().all(), f"Faltan retornos en el pliegue {pliegue}"

    capital = capital_inicial * (1 + retornos).cumprod()
    maximo_acumulado = capital.cummax().clip(lower=capital_inicial)
    drawdown = capital.div(maximo_acumulado).sub(1)

    resultados_buy_hold.append({
        "pliegue": pliegue,
        "inicio_validacion": fechas_val[0].date(),
        "fin_validacion": fechas_val[-1].date(),
        "capital_final_usd": capital.iloc[-1],
        "rendimiento_acumulado_pct": (capital.iloc[-1] / capital_inicial - 1) * 100,
        "max_drawdown_pct": -drawdown.min() * 100,
    })

resultados_buy_hold = pd.DataFrame(resultados_buy_hold)
display(resultados_buy_hold.round(2))

,pliegue,inicio_validacion,fin_validacion,capital_final_usd,rendimiento_acumulado_pct,max_drawdown_pct
0,1,2012-11-13,2014-12-29,15185.46,51.85,33.37
1,2,2014-12-30,2017-02-13,12529.72,25.30,30.44
2,3,2017-02-14,2019-04-01,14819.32,48.19,38.52
3,4,2019-04-02,2021-05-14,26622.69,166.23,31.43
4,5,2021-05-17,2023-06-30,15420.56,54.21,30.91


## Candidatos tabulares

In [39]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier


def evaluar_candidato(estimador, excluir_columnas=()):
    resultados = []

    X_evaluacion = X_desarrollo.drop(columns=list(excluir_columnas))
    numericas_evaluacion = [
        columna
        for columna in columnas_numericas
        if columna in X_evaluacion.columns
    ]

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_evaluacion), start=1
    ):
        X_train = X_evaluacion.iloc[idx_train]
        X_val = X_evaluacion.iloc[idx_val]
        vol_train = vol_desarrollo.iloc[idx_train]
        vol_val = vol_desarrollo.iloc[idx_val]

        p80 = vol_train.quantile(0.80)
        y_train = vol_train.gt(p80).astype(int)
        y_val = vol_val.gt(p80).astype(int)

        pipeline = crear_pipeline(clone(estimador), numericas_evaluacion)

        inicio = perf_counter()
        pipeline.fit(X_train, y_train)
        segundos_ajuste = perf_counter() - inicio

        auc_train = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
        auc_val = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

        resultados.append({
            "pliegue": pliegue,
            "fin_train": X_train.index[-1].date(),
            "inicio_validacion": X_val.index[0].date(),
            "fin_validacion": X_val.index[-1].date(),
            "p80_train": p80,
            "roc_auc_train": auc_train,
            "roc_auc_val": auc_val,
            "brecha_roc_auc": auc_train - auc_val,
            "segundos_ajuste": segundos_ajuste,
        })

    return pd.DataFrame(resultados)

In [40]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

candidatos = {
    "logistica": ("lineal_regularizado", LogisticRegression(random_state=SEMILLA)),
    "random_forest": ("bagging", RandomForestClassifier(random_state=SEMILLA)),
    "boosting": ("boosting", HistGradientBoostingClassifier(random_state=SEMILLA)),
    "mlp": ("red_neuronal", MLPClassifier(random_state=SEMILLA)),
    "svm": ("maquina_vectores_soporte", SVC(probability=True, random_state=SEMILLA)),
}

resultados_candidatos = {
    nombre: evaluar_candidato(estimador)
    for nombre, (_, estimador) in candidatos.items()
}
ranking = pd.DataFrame([
    {
        "modelo": type(candidatos[nombre][1]).__name__,
        "roc_auc_train": resultados["roc_auc_train"].mean(),
        "roc_auc_cv": resultados["roc_auc_val"].mean(),
        "std_cv": resultados["roc_auc_val"].std(ddof=1),
        "brecha": resultados["brecha_roc_auc"].mean(),
        "segundos_cv": resultados["segundos_ajuste"].sum(),
    }
    for nombre, resultados in resultados_candidatos.items()
]).sort_values("roc_auc_cv", ascending=False)
display(ranking.round(4))

bosque = candidatos["random_forest"][1]
resultados_bosque = resultados_candidatos["random_forest"]

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and 

,modelo,roc_auc_train,roc_auc_cv,std_cv,brecha,segundos_cv
1,RandomForestClassifier,1.0000,0.7226,0.0480,0.2774,6.8115
0,LogisticRegression,0.7811,0.7099,0.0679,0.0712,0.8680
4,SVC,0.9314,0.6995,0.0401,0.2319,5.3804
2,HistGradientBoostingClassifier,1.0000,0.6514,0.0495,0.3486,4.5778
3,MLPClassifier,0.9653,0.6361,0.0750,0.3292,13.4764


In [41]:
# Ablación de Random Forest

columnas_volatilidad_historica = [
    "VOL_HIST_5",
    "VOL_HIST_20",
    "VOL_HIST_60",
    "VOL_RATIO_5_20",
]

resultados_bosque_sin_vol = evaluar_candidato(
    bosque,
    excluir_columnas=columnas_volatilidad_historica,
)

ablacion_bosque = resultados_bosque[["pliegue", "roc_auc_val"]].merge(
    resultados_bosque_sin_vol[["pliegue", "roc_auc_val"]],
    on="pliegue",
    suffixes=("_completo", "_sin_volatilidad"),
    validate="one_to_one",
)

ablacion_bosque["aporte_grupo_auc"] = (
    ablacion_bosque["roc_auc_val_completo"]
    - ablacion_bosque["roc_auc_val_sin_volatilidad"]
)

display(ablacion_bosque.round(4))
aporte_medio = ablacion_bosque["aporte_grupo_auc"].mean()
print(f"Aporte medio del grupo: {aporte_medio:+.4f}")

,pliegue,roc_auc_val_completo,roc_auc_val_sin_volatilidad,aporte_grupo_auc
0,1,0.6482,0.6207,0.0275
1,2,0.7119,0.7025,0.0094
2,3,0.7429,0.7799,-0.0371
3,4,0.7321,0.7339,-0.0018
4,5,0.7778,0.7796,-0.0018


Aporte medio del grupo: -0.0008


La ablación compara las cuatro variables de volatilidad histórica como grupo. La corrida de revisión anterior obtuvo 0,7226 con el grupo y 0,7233 sin él (MLflow: `f174d42bde9d47cdbb18c14be83d11e4`). La tabla anterior se calcula con los datos de la ejecución actual.

## Registro de líneas base y candidatos

In [42]:
import subprocess
from mlflow.models import infer_signature

def registrar_candidato(estimador, resultados, *, nombre_corrida, familia, nombre_pipeline):
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()
    cambios_pendientes = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
    ).strip())

    hash_dataset = sha256(
        entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
    ).hexdigest()

    p80_desarrollo = float(vol_desarrollo.quantile(0.80))
    y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

    pipeline = crear_pipeline(clone(estimador), columnas_numericas)
    inicio = perf_counter()
    pipeline.fit(X_desarrollo, y_desarrollo)
    segundos_reajuste = perf_counter() - inicio

    parametros = estimador.get_params(deep=False)
    ejemplo = X_desarrollo.head(5).copy()
    firma = infer_signature(ejemplo, pipeline.predict_proba(ejemplo))

    with mlflow.start_run(run_name=nombre_corrida) as corrida:
        mlflow.set_tags({
            "fase": "candidato",
            "familia": familia,
            "git_commit": commit,
            "git_cambios_sin_commit": str(cambios_pendientes).lower(),
        })

        mlflow.log_params({
            **datos_corrida,
            **{f"modelo__{nombre}": valor for nombre, valor in parametros.items()},
            "sha256_dataset_desarrollo": hash_dataset,
            "modelo": type(estimador).__name__,
            "semilla": parametros.get("random_state", "no_aplica"),
            "n_pliegues": cv_temporal.n_splits,
            "gap": cv_temporal.gap,
            "inicio_holdout": inicio_prueba.date().isoformat(),
            "p80_desarrollo": p80_desarrollo,
        })

        mlflow.log_metrics({
            "roc_auc_train_media_cv": float(resultados["roc_auc_train"].mean()),
            "roc_auc_val_media_cv": float(resultados["roc_auc_val"].mean()),
            "roc_auc_val_std_cv": float(resultados["roc_auc_val"].std(ddof=1)),
            "brecha_roc_auc_cv": float(resultados["brecha_roc_auc"].mean()),
            "segundos_ajuste_cv_total": float(resultados["segundos_ajuste"].sum()),
            "segundos_reajuste_desarrollo": segundos_reajuste,
        })

        mlflow.log_text(resultados.to_csv(index=False), "resultados_por_pliegue.csv")
        modelo = mlflow.sklearn.log_model(
            pipeline,
            name=nombre_pipeline,
            signature=firma,
            input_example=ejemplo,
            pyfunc_predict_fn="predict_proba",
        )

    print("Run ID:", corrida.info.run_id)
    print("URI del modelo:", modelo.model_uri)
    return corrida.info.run_id

In [48]:
# Celda que se activa solo si REGISTRAR_CANDIDATOS es True para grabar en el MLFlow

if REGISTRAR_CANDIDATOS:
    experimento = mlflow.set_experiment(NOMBRE_EXPERIMENTO)

    import subprocess
    from time import perf_counter

    import mlflow.sklearn
    from mlflow.models import infer_signature

    commit_git = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    cambios_sin_commit = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "-uno"],
        cwd=raiz,
        text=True,
    ).strip())

    hash_desarrollo = sha256(
        entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
    ).hexdigest()

    p80_desarrollo = vol_desarrollo.quantile(0.80)
    y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

    dummy_reajustado = crear_pipeline(
        DummyClassifier(strategy="most_frequent"),
        columnas_numericas,
    )

    inicio = perf_counter()
    dummy_reajustado.fit(X_desarrollo, y_desarrollo)
    segundos_reajuste = perf_counter() - inicio

    ejemplo = X_desarrollo.head(5)
    firma = infer_signature(ejemplo, dummy_reajustado.predict_proba(ejemplo))

    auc_train = resultados_dummy["roc_auc_train"].mean()
    auc_val = resultados_dummy["roc_auc_val"].mean()

    with mlflow.start_run(run_name="baseline_dummy_most_frequent") as corrida:
        mlflow.set_tags({
            "fase": "baseline",
            "familia": "DummyClassifier",
            "git_commit": commit_git,
            "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
        })

        mlflow.log_params({
            **datos_corrida,
            "sha256_dataset_desarrollo": hash_desarrollo,
            "estrategia": "most_frequent",
            "semilla": "no_aplica",
            "n_pliegues": cv_temporal.n_splits,
            "gap": HORIZONTE,
            "inicio_holdout": inicio_prueba.date().isoformat(),
            "p80_desarrollo": float(p80_desarrollo),
        })

        mlflow.log_metrics({
            "roc_auc_train_media_cv": float(auc_train),
            "roc_auc_val_media_cv": float(auc_val),
            "roc_auc_val_std_cv": float(resultados_dummy["roc_auc_val"].std(ddof=1)),
            "brecha_roc_auc_cv": float(auc_train - auc_val),
            "segundos_ajuste_cv_total": float(resultados_dummy["segundos_ajuste"].sum()),
            "segundos_reajuste_desarrollo": segundos_reajuste,
        })

        mlflow.log_text(
            resultados_dummy.to_csv(index=False),
            "resultados_por_pliegue.csv",
        )

        modelo_registrado = mlflow.sklearn.log_model(
            dummy_reajustado,
            name="pipeline_dummy",
            signature=firma,
            input_example=ejemplo,
            pyfunc_predict_fn="predict_proba",
        )

    print("Run ID:", corrida.info.run_id)
    print("URI del modelo:", modelo_registrado.model_uri)

    from hashlib import sha256
    import subprocess

    commit_git = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    cambios_sin_commit = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
    ).strip())

    hash_desarrollo = sha256(
        entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
    ).hexdigest()

    auc_train = resultados_persistencia["roc_auc_train"].mean()
    auc_val = resultados_persistencia["roc_auc_val"].mean()
    p80_desarrollo = float(vol_desarrollo.quantile(0.80))

    with mlflow.start_run(run_name="baseline_persistencia_vol5") as corrida:
        mlflow.set_tags({
            "fase": "baseline",
            "familia": "persistencia",
            "git_commit": commit_git,
            "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
        })

        mlflow.log_params({
            **datos_corrida,
            "sha256_dataset_desarrollo": hash_desarrollo,
            "estrategia": "volatilidad_historica_como_pronostico",
            "feature_score": "VOL_HIST_5",
            "semilla": "no_aplica",
            "n_pliegues": cv_temporal.n_splits,
            "gap": HORIZONTE,
            "inicio_holdout": inicio_prueba.date().isoformat(),
            "p80_desarrollo": p80_desarrollo,
        })

        mlflow.log_metrics({
            "roc_auc_train_media_cv": float(auc_train),
            "roc_auc_val_media_cv": float(auc_val),
            "roc_auc_val_std_cv": float(
                resultados_persistencia["roc_auc_val"].std(ddof=1)
            ),
            "brecha_roc_auc_cv": float(auc_train - auc_val),
        })

        mlflow.log_text(
            resultados_persistencia.to_csv(index=False),
            "resultados_por_pliegue.csv",
        )

        mlflow.log_dict({
            "score": "VOL_HIST_5",
            "columna_para_estimar_umbral": "VOL_FUTURA_5",
            "cuantil": 0.80,
            "condicion_alerta": "score > umbral",
            "p80_desarrollo": p80_desarrollo,
        }, "regla_persistencia.json")

    print("Run ID:", corrida.info.run_id)

    commit_git = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    cambios_sin_commit = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
    ).strip())

    with mlflow.start_run(run_name="baseline_buy_hold") as corrida:
        mlflow.set_tags({
            "fase": "baseline",
            "familia": "regla_negocio",
            "git_commit": commit_git,
            "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
        })

        mlflow.log_params({
            **datos_corrida,
            "sha256_dataset_desarrollo": hash_desarrollo,
            "estrategia": "buy_and_hold",
            "capital_inicial_usd": capital_inicial,
            "exposicion_aapl": 1.0,
            "costos_transaccion": 0.0,
            "retorno": "cierre_ajustado_t_a_cierre_ajustado_t_mas_1",
            "reinicio_capital_por_pliegue": True,
            "n_pliegues": cv_temporal.n_splits,
            "gap": HORIZONTE,
            "inicio_holdout": inicio_prueba.date().isoformat(),
        })

        mlflow.log_metrics({
            "capital_final_usd_media_cv": float(
                resultados_buy_hold["capital_final_usd"].mean()
            ),
            "max_drawdown_pct_media_cv": float(
                resultados_buy_hold["max_drawdown_pct"].mean()
            ),
        })

        mlflow.log_text(
            resultados_buy_hold.to_csv(index=False),
            "resultados_por_pliegue.csv",
        )

    print("Run ID:", corrida.info.run_id)

    ids_candidatos = [
        registrar_candidato(
            estimador, resultados_candidatos[nombre],
            nombre_corrida=f"candidato_{nombre}_default_revision",
            familia=familia, nombre_pipeline=f"pipeline_{nombre}",
        )
        for nombre, (familia, estimador) in candidatos.items()
    ]
    guardar_ids({
        "sha256_dataset_desarrollo": hash_desarrollo,
        "sha256_snapshot_obt": datos_corrida["sha256_snapshot_obt"],
        "candidatos": ids_candidatos,
    })
    print("IDs de candidatos:", ids_candidatos)
else:
    print('Registro de candidatos desactivado.')

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 17:43:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run baseline_dummy_most_frequent at: http://127.0.0.1:5000/#/experiments/1/runs/6700d87ef20b4cff8f821770cfd49aba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 6700d87ef20b4cff8f821770cfd49aba
URI del modelo: models:/m-8ecf341d1a054756917bc73014ce272c
🏃 View run baseline_persistencia_vol5 at: http://127.0.0.1:5000/#/experiments/1/runs/7135fda72e6a44469d71b3481709275a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 7135fda72e6a44469d71b3481709275a
🏃 View run baseline_buy_hold at: http://127.0.0.1:5000/#/experiments/1/runs/2cf5984c8410401b9c00f1d4fef66302
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 2cf5984c8410401b9c00f1d4fef66302


c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 17:46:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_logistica_default_revision at: http://127.0.0.1:5000/#/experiments/1/runs/f7e8fd5b27da4cc890134287aae13786
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: f7e8fd5b27da4cc890134287aae13786
URI del modelo: models:/m-f6a8b462401640f8990186ef9a638d99


c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 17:51:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_random_forest_default_revision at: http://127.0.0.1:5000/#/experiments/1/runs/73cd1c64ffd24467bdfda29963d97eab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow-artifacts/artifacts/1/models/m-e0560fd912d84b1781fb52cf35214be9/artifacts/model.pkl failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/1/models/m-e0560fd912d84b1781fb52cf35214be9/artifacts/model.pkl (Caused by ProtocolError('Connection aborted.', TimeoutError('timed out')))

In [49]:
from mlflow.tracking import MlflowClient

cliente_diagnostico = MlflowClient(tracking_uri="http://127.0.0.1:5000")

for nombre, run_id in {
    "Regresión logística": "f7e8fd5b27da4cc890134287aae13786",
    "Random Forest": "73cd1c64ffd24467bdfda29963d97eab",
}.items():
    corrida = cliente_diagnostico.get_run(run_id)
    print(f"{nombre}: {corrida.info.status} | {run_id}")

print("IDs acumulados:", globals().get("ids_candidatos", []))
print("Resultados de CV en memoria:", "resultados_candidatos" in globals())

Regresión logística: FINISHED | f7e8fd5b27da4cc890134287aae13786
Random Forest: FAILED | 73cd1c64ffd24467bdfda29963d97eab
IDs acumulados: []
Resultados de CV en memoria: True


In [50]:
_, bosque_revision = candidatos["random_forest"]

id_bosque_revision = registrar_candidato(
    bosque_revision,
    resultados_candidatos["random_forest"],
    nombre_corrida="candidato_random_forest_default_revision_reintento",
    familia="bagging",
    nombre_pipeline="pipeline_random_forest",
)

print("Random Forest registrado:", id_bosque_revision, flush=True)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 18:41:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_random_forest_default_revision_reintento at: http://127.0.0.1:5000/#/experiments/1/runs/432e8271a62e42389de9d59e8d934d48
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 432e8271a62e42389de9d59e8d934d48
URI del modelo: models:/m-22664c1217394b0cbacc233b1214016f
Random Forest registrado: 432e8271a62e42389de9d59e8d934d48


In [51]:
if "ids_revision_por_nombre" not in globals():
    ids_revision_por_nombre = {
        "logistica": "f7e8fd5b27da4cc890134287aae13786",
        "random_forest": "432e8271a62e42389de9d59e8d934d48",
    }

for nombre in ("boosting", "mlp", "svm"):
    if nombre in ids_revision_por_nombre:
        continue

    familia, estimador = candidatos[nombre]
    print(f"Registrando {nombre}...", flush=True)

    ids_revision_por_nombre[nombre] = registrar_candidato(
        estimador,
        resultados_candidatos[nombre],
        nombre_corrida=f"candidato_{nombre}_default_revision",
        familia=familia,
        nombre_pipeline=f"pipeline_{nombre}",
    )

ids_candidatos = [ids_revision_por_nombre[nombre] for nombre in candidatos]

guardar_ids({
    "sha256_dataset_desarrollo": hash_desarrollo,
    "sha256_snapshot_obt": datos_corrida["sha256_snapshot_obt"],
    "candidatos": ids_candidatos,
})
print("IDs de candidatos:", ids_candidatos, flush=True)

Registrando boosting...


c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 18:44:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_boosting_default_revision at: http://127.0.0.1:5000/#/experiments/1/runs/d6f267f4c60346c58462eaa0af75ce95
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: d6f267f4c60346c58462eaa0af75ce95
URI del modelo: models:/m-a04511a5efe14a09b1b37b99c648c3b1
Registrando mlp...


c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-mi

🏃 View run candidato_mlp_default_revision at: http://127.0.0.1:5000/#/experiments/1/runs/95f5bca094da491aacf43ce1da4349ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 95f5bca094da491aacf43ce1da4349ce
URI del modelo: models:/m-5a614f08b3b744399cd9850650be872c
Registrando svm...


c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 18:50:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_svm_default_revision at: http://127.0.0.1:5000/#/experiments/1/runs/b6350a1f8f5945668664408312e7d68f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: b6350a1f8f5945668664408312e7d68f
URI del modelo: models:/m-11c4a247a570438a8018d37a9e262c58
IDs de candidatos: ['f7e8fd5b27da4cc890134287aae13786', '432e8271a62e42389de9d59e8d934d48', 'd6f267f4c60346c58462eaa0af75ce95', '95f5bca094da491aacf43ce1da4349ce', 'b6350a1f8f5945668664408312e7d68f']


## TimesFM 3.0

La corrida `f29c26f203de41c8a81cb03b01b4cef8` obtuvo ROC-AUC 0,5985 ± 0,1359 en 2.675 fechas. Los CSV por fecha y pliegue están en sus artefactos. La inferencia usa el entorno Windows separado `%LOCALAPPDATA%/venvs/pset3-timesfm` con TimesFM 3.0.2 y PyTorch. Habilita la inferencia para reproducirla con la extracción actual.


Resultados guardados de la corrida anterior:

```text
pliegue  n_fechas  roc_auc_val
0        1       535       0.5120
1        2       535       0.6138
2        3       535       0.7549
3        4       535       0.6937
4        5       535       0.4180
```

In [ ]:
if RECALCULAR_TIMESFM:
    ## Modelo con todas las Features (22 mins approx)

    import json
    import os
    import subprocess
    from pathlib import Path
    from tempfile import TemporaryDirectory

    import pandas as pd
    from sklearn.metrics import roc_auc_score

    python_timesfm = (
        Path(os.environ["LOCALAPPDATA"])
        / "venvs" / "pset3-timesfm" / "Scripts" / "python.exe"
    )
    modelo_timesfm = "google/timesfm-3.0-pytorch"
    longitud_contexto = 512
    tamano_lote = 8

    retornos = features["RET_LAG_0"].dropna().sort_index()
    contextos = []
    filas_validacion = []

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_desarrollo), start=1
    ):
        p80 = vol_desarrollo.iloc[idx_train].quantile(0.80)

        for fecha, volatilidad in vol_desarrollo.iloc[idx_val].items():
            contexto = retornos.loc[:fecha].tail(longitud_contexto)
            assert len(contexto) == longitud_contexto and contexto.index[-1] == fecha

            contextos.append(contexto.tolist())
            filas_validacion.append({
                "pliegue": pliegue,
                "fecha_alerta": fecha.date().isoformat(),
                "p80_train": float(p80),
                "etiqueta_real": int(volatilidad > p80),
            })

    codigo_timesfm = """
    import json
    import sys
    from pathlib import Path
    from time import perf_counter

    import numpy as np
    from timesfm3 import ModelConfig, TimesFM3Evaluator

    entrada, salida, checkpoint, lote = sys.argv[1:]
    contextos = json.loads(Path(entrada).read_text(encoding="utf-8"))
    lote = int(lote)

    print("Cargando TimesFM en CPU...", flush=True)
    modelo = TimesFM3Evaluator(ModelConfig(
        checkpoint_path=checkpoint,
        per_core_batch_size=lote,
        device="cpu",
    ))

    puntajes = []
    inicio = perf_counter()

    for desde in range(0, len(contextos), lote):
        series = [np.asarray(x, dtype=np.float32) for x in contextos[desde:desde + lote]]
        resultados = modelo.predict_batch(
            series,
            horizon=5,
            return_quantiles=False,
            use_symmetric_averaging=False,
        )
        for resultado in resultados:
            pronostico = np.asarray(resultado.forecast)
            assert pronostico.shape == (5,) and np.isfinite(pronostico).all()
            puntajes.append(float(pronostico.std(ddof=1)))

        if desde == 0 or (desde // lote + 1) % 10 == 0:
            print(f"Pronosticos completos: {len(puntajes)}/{len(contextos)}", flush=True)

    assert len(puntajes) == len(contextos)
    Path(salida).write_text(json.dumps({
        "puntajes": puntajes,
        "segundos_inferencia": perf_counter() - inicio,
    }), encoding="utf-8")
    """

    with TemporaryDirectory() as carpeta:
        entrada = Path(carpeta) / "contextos.json"
        salida = Path(carpeta) / "puntajes.json"
        entrada.write_text(json.dumps(contextos, allow_nan=False), encoding="utf-8")

        with subprocess.Popen(
            [
                str(python_timesfm), "-u", "-c", codigo_timesfm,
                str(entrada), str(salida), modelo_timesfm, str(tamano_lote),
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
        ) as proceso:
            for linea in proceso.stdout:
                print(linea.rstrip(), flush=True)
            if proceso.wait() != 0:
                raise RuntimeError("Falló TimesFM. Revisa el mensaje anterior.")

        evaluacion = json.loads(salida.read_text(encoding="utf-8"))

    detalle_timesfm = pd.DataFrame(filas_validacion)
    detalle_timesfm["puntaje_volatilidad"] = evaluacion["puntajes"]

    resultados_timesfm = pd.DataFrame([
        {
            "pliegue": pliegue,
            "n_fechas": len(datos),
            "roc_auc_val": roc_auc_score(
                datos["etiqueta_real"], datos["puntaje_volatilidad"]
            ),
        }
        for pliegue, datos in detalle_timesfm.groupby("pliegue")
    ])

    display(resultados_timesfm.round(4))
    auc_timesfm = resultados_timesfm["roc_auc_val"]
    print(f"ROC-AUC CV: {auc_timesfm.mean():.4f} ± {auc_timesfm.std(ddof=1):.4f}")
    print(f"Tiempo de inferencia: {evaluacion['segundos_inferencia']:.1f} s")
else:
    print('TimesFM: se conserva la referencia a la corrida previa.')

TimesFM: se conserva la referencia a la corrida previa.


In [ ]:
if RECALCULAR_TIMESFM and REGISTRAR_TIMESFM:
    import mlflow
    from mlflow.tracking import MlflowClient

    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    cliente = MlflowClient()

    nombre_experimento = "alerta-volatilidad-aapl-modelos-fundacionales"
    experimento = cliente.get_experiment_by_name(nombre_experimento)
    id_experimento = (
        experimento.experiment_id
        if experimento is not None
        else cliente.create_experiment(nombre_experimento)
    )

    with mlflow.start_run(
        experiment_id=id_experimento,
        run_name="timesfm_3_0_zero_shot",
        tags={"fase": "candidatos", "familia_modelo": "fundacional"},
    ) as corrida:
        mlflow.log_params({
            **datos_corrida,
            "checkpoint": modelo_timesfm,
            "dispositivo": "cpu",
            "contexto": longitud_contexto,
            "horizonte": 5,
            "tamano_lote": tamano_lote,
            "pliegues": cv_temporal.n_splits,
            "gap": cv_temporal.gap,
            "puntaje": "desviacion_estandar_de_5_pronosticos_puntuales",
        })
        mlflow.log_metrics({
            "roc_auc_cv_media": float(auc_timesfm.mean()),
            "roc_auc_cv_std": float(auc_timesfm.std(ddof=1)),
            "segundos_inferencia": evaluacion["segundos_inferencia"],
            "n_fechas_evaluadas": len(detalle_timesfm),
        })
        mlflow.log_text(resultados_timesfm.to_csv(index=False), "timesfm_por_pliegue.csv")
        mlflow.log_text(detalle_timesfm.to_csv(index=False), "timesfm_por_fecha.csv")

    print("Run ID:", corrida.info.run_id)

## Claude · Prueba exploratoria por prompting

Se usó la versión gratuita de Claude. Se guardan los puntajes devueltos para tres lotes de 20 fechas. Cada lote se seleccionó a intervalos regulares dentro del pliegue, sin utilizar sus etiquetas. La comparación de ROC-AUC usa las mismas fechas para Claude y persistencia. La versión exacta del modelo no quedó registrada.


Resultados históricos del piloto: pliegue 1, 0,7333 frente a 0,6267 (5 positivos); pliegue 2, 0,5980 frente a 0,5882 (3 positivos).

In [ ]:
import numpy as np

puntajes_claude = {
    1: [0.55, 0.60, 0.60, 0.50, 0.45, 0.40, 0.30, 0.35, 0.40, 0.30,
        0.25, 0.30, 0.25, 0.30, 0.25, 0.20, 0.20, 0.20, 0.20, 0.25],
    2: [0.25, 0.45, 0.40, 0.40, 0.35, 0.40, 0.60, 0.60, 0.45, 0.40,
        0.55, 0.50, 0.35, 0.35, 0.25, 0.25, 0.30, 0.25, 0.15, 0.20],
    3: [0.15, 0.15, 0.15, 0.20, 0.20, 0.25, 0.20, 0.25, 0.20, 0.40,
        0.50, 0.55, 0.40, 0.20, 0.30, 0.50, 0.70, 0.80, 0.60, 0.40],
}
hash_claude = "e2fe712d5cfc35cd58969d960b4be048ff32442b93e2bc610e8d955f980f1b60"
if hash_desarrollo != hash_claude:
    print("La huella de desarrollo difiere. Consulta las métricas históricas en el memo o notebook original.")
else:
    filas_claude = []
    prompts_claude = {}
    for pliegue, (idx_train, idx_val) in enumerate(cv_temporal.split(X_desarrollo), 1):
        if pliegue not in puntajes_claude:
            break
        puntajes = puntajes_claude[pliegue]
        indices = idx_val[np.linspace(0, len(idx_val) - 1, len(puntajes), dtype=int)]
        fechas = X_desarrollo.index[indices]
        p80 = vol_desarrollo.iloc[idx_train].quantile(CUANTIL_TARGET)
        y = vol_desarrollo.loc[fechas].gt(p80)
        casos = [
            {"id": f"P{pliegue}-{i:02d}", "retornos_pct":
             features.loc[:fecha, "RET_LAG_0"].dropna().tail(60).round(6).tolist()}
            for i, fecha in enumerate(fechas, 1)
        ]
        prompts_claude[pliegue] = (
            f"Estima por caso la probabilidad de que la desviación estándar muestral de "
            f"los próximos cinco retornos diarios supere {p80:.6f} %. Usa solo los 60 "
            "retornos en porcentaje de cada caso. Devuelve solo un array JSON con "
            "id y probabilidad_volatilidad_alta (número entre 0 y 1).\n"
            + json.dumps(casos, allow_nan=False)
        )
        filas_claude.append({
            "pliegue": pliegue, "fechas": len(y), "positivos": int(y.sum()),
            "auc_claude": roc_auc_score(y, puntajes) if y.nunique() == 2 else None,
            "auc_persistencia": roc_auc_score(y, features.loc[fechas, "VOL_HIST_5"])
                if y.nunique() == 2 else None,
        })
    display(pd.DataFrame(filas_claude).round(4))

,pliegue,fechas,positivos,auc_claude,auc_persistencia
0,1,20,5,0.7333,0.6267
1,2,20,3,0.5980,0.5882
2,3,20,4,0.5000,0.5469
